# Chapter 13 - Homology

**Source Span.** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Chapter 13, printed pp. 339-380; assigned PDF pp. 357-398. Source inspection note: `pdftotext -f 357 -l 398` was sampled first as assigned, but in this local file it begins at printed p. 344 and continues into Appendix A. To cover the printed chapter span exactly, I also inspected the adjusted physical extraction `pdftotext -f 352 -l 393`, which aligns with printed pp. 339-380.

## Chapter Goal

Homology turns spaces into computable algebra by replacing loops and higher-dimensional surfaces with signed chains. The central question of this notebook is: **which cycles fail to be boundaries, and which features of a space are detected by that failure?** We will work with finite models that imitate the singular theory. The finite models are not a replacement for singular chains, whose chain groups are enormous, but they expose the same algebraic moves: boundary maps satisfy `d*d = 0`, continuous maps induce chain maps, chain homotopies prove homotopy invariance, Mayer-Vietoris assembles local data, cellular chains compute finite CW complexes, Euler characteristic is recovered from Betti numbers, and cohomology reverses arrows by testing chains with functions.

The source chapter develops the full singular theory. Here we keep the geometry visible. A cycle is shown as an object with no remaining signed boundary. A boundary matrix records how oriented faces cancel. A homotopy is shown as a prism whose side faces are the correction term. Mayer-Vietoris is shown on a circle cover where the overlap has two components, creating the missing one-dimensional class. CW homology is shown as a compact algebraic shadow of attaching cells. Cohomology is previewed as the dual bookkeeping system: it extracts numbers from cycles and pulls information backward along maps.


## Computational Translation Guide

| Topological phrase | Computational object in this notebook | What to inspect |
| --- | --- | --- |
| Singular `p`-simplex | An oriented `p`-simplex or a small finite proxy | Its ordered vertices and signed faces |
| Singular `p`-chain | Integer vector in a chain group `C_p` | Coefficients of oriented pieces |
| Boundary operator | Sparse integer matrix `d_p: C_p -> C_{p-1}` | Column sums and cancellation patterns |
| Cycle | Vector in `ker(d_p)` | Boundary matrix sends it to zero |
| Boundary | Vector in `im(d_{p+1})` | It is produced by a higher chain |
| Homology group | `ker(d_p) / im(d_{p+1})` | Betti rank plus possible torsion |
| Chain map | Matrices commuting with boundaries | `d F = F d` |
| Chain homotopy | A correction matrix or prism chain `h` | `d h + h d = G - F` |
| Mayer-Vietoris | Exact sequence from `U`, `V`, and `U cap V` | Overlap classes that glue to global cycles |
| Finite CW complex | Cellular chain complex | Boundary maps determined by attaching maps |
| Cohomology | Dual vector spaces and transpose maps | Arrows reverse: cochains pull back |

This guide deliberately uses small exact integer matrices. Singular homology is defined for all continuous simplices in a space, but the chapter's computational power comes from proving that the giant singular complex can often be replaced by smaller chain complexes without changing homology.


## Library Routing

| Concept | Representation | Library | Why this route fits |
| --- | --- | --- | --- |
| Boundary of a simplex and `d*d = 0` | Exact integer boundary matrices plus a labeled triangle | `sympy`, `matplotlib` | The theorem is algebraic cancellation with a geometric sign pattern. |
| Proof dependency and chain-complex flow | Directed graph and rank flow | `networkx`, `plotly` | Homology proofs are diagrams of maps; interactivity helps inspect dimensions. |
| Homotopy invariance | Prism triangulation and chain-homotopy identity | `plotly`, exact dictionary arithmetic | The geometric proof is a prism whose side faces cancel algebraically. |
| Mayer-Vietoris for a circle cover | Circle arcs and an exact-sequence rank check | `matplotlib`, `sympy` | The overlap has two components; one kernel class becomes the circle's `H_1`. |
| CW homology and Euler characteristic | Cellular chain tables, Betti ranks, and bar checks | `pandas`, `sympy`, `matplotlib` | Finite CW complexes are naturally finite chain complexes. |
| Cohomology preview | Reversed-arrow diagram and dual rank check | `networkx`, `sympy`, `matplotlib` | Cohomology is contravariant; the diagram makes the reversal explicit. |

The topology catalog recommends topology-oriented libraries such as Gudhi or Ripser when persistence or sampled shape recovery is the topic. Chapter 13 is primarily about exact chain complexes and functorial algebra, so exact integer linear algebra and graph/diagram tools carry the pedagogy more directly than a persistent-homology pipeline.


## Visual Storyboard

The notebook saves the storyboard as `artifacts/chapter-13-homology/checks/visual-storyboard.json`. Each visual has an inspection target and a validation. The purpose is not to decorate the chapter; each artifact is a finite model of a theorem move from singular homology.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import Markdown, display


def locate_book_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        direct = candidate
        nested = candidate / "Introduction-to-Topological-Manifolds"
        if (direct / "AGENTS.md").exists() and (direct / "source_map.json").exists() and (direct / "utils").exists():
            return direct
        if (nested / "AGENTS.md").exists() and (nested / "source_map.json").exists() and (nested / "utils").exists():
            return nested
    raise RuntimeError("Could not locate Introduction-to-Topological-Manifolds root")


BOOK_ROOT = locate_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, chapter_artifact_root, display_artifact, save_csv, save_json, save_matplotlib, save_plotly_html
from utils.topology import euler_characteristic, simplex_boundary_squared_zero
from utils.validation import image_stats, relative

UNIT_KEY = "chapter-13-homology"
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / "figures"
HTML = ARTIFACT_ROOT / "html"
CHECKS = ARTIFACT_ROOT / "checks"
TABLES = ARTIFACT_ROOT / "tables"

plt.rcParams.update({"figure.dpi": 140, "font.size": 10})
print(f"BOOK_ROOT = {BOOK_ROOT}")
print(f"ARTIFACT_ROOT = {relative(ARTIFACT_ROOT)}")


In [ ]:
storyboard = {
    "chapter_goal": "Make singular homology computable through finite chain-complex models, exact identities, and visual proof scaffolds.",
    "source_span_read": {
        "assigned": "printed pp. 339-380; PDF pp. 357-398",
        "pdftotext_note": "Assigned extraction was sampled; adjusted physical pages 352-393 aligned with printed pp. 339-380 in this local PDF."
    },
    "visual_sequence": [
        {
            "concept": "singular boundary and boundary-squared-zero",
            "artifact": "figures/singular-boundary-cancellation.png",
            "representation": "oriented triangle plus exact boundary matrices",
            "library": "matplotlib + sympy",
            "inspection_target": "opposite signs on repeated codimension-two faces",
            "validation": "d1*d2 is the zero matrix"
        },
        {
            "concept": "proof dependencies from chains to invariants",
            "artifact": "figures/homology-proof-dependency-graph.png",
            "representation": "directed proof graph",
            "library": "networkx + matplotlib",
            "inspection_target": "which algebraic facts feed homotopy invariance, Mayer-Vietoris, and Euler characteristic",
            "validation": "graph is acyclic and reaches all required chapter themes"
        },
        {
            "concept": "homotopy invariance",
            "artifact": "html/prism-chain-homotopy.html",
            "representation": "triangulated prism for an edge homotopy",
            "library": "plotly",
            "inspection_target": "top minus bottom remains after side faces cancel",
            "validation": "exact chain dictionary satisfies d h + h d = top - bottom"
        },
        {
            "concept": "Mayer-Vietoris on the circle",
            "artifact": "figures/mayer-vietoris-circle-cover.png",
            "representation": "two connected arcs with a two-component overlap",
            "library": "matplotlib + sympy",
            "inspection_target": "one overlap difference class generates H1(S1)",
            "validation": "rank kernel of H0(U cap V)->H0(U)oplusH0(V) is one"
        },
        {
            "concept": "CW homology and Euler characteristic",
            "artifact": "figures/cw-euler-betti-dashboard.png",
            "representation": "cellular chain model table and Euler rank bars",
            "library": "pandas + sympy + matplotlib",
            "inspection_target": "cell counts and Betti ranks give the same Euler characteristic",
            "validation": "alternating cell count equals alternating Betti rank for each example"
        },
        {
            "concept": "cohomology preview",
            "artifact": "figures/cohomology-dual-arrows.png",
            "representation": "chain arrows and reversed cochain arrows",
            "library": "networkx + matplotlib",
            "inspection_target": "pullback reverses direction while dimensions over characteristic-zero fields match Betti ranks",
            "validation": "torus cohomology dimensions equal torus homology ranks over Q"
        }
    ],
    "artifact_plan": {
        "figures": [
            "singular-boundary-cancellation.png",
            "homology-proof-dependency-graph.png",
            "mayer-vietoris-circle-cover.png",
            "cw-euler-betti-dashboard.png",
            "cw-attachment-lab.png",
            "cohomology-dual-arrows.png"
        ],
        "html": ["prism-chain-homotopy.html", "chain-complex-rank-flow.html"],
        "checks": [
            "visual-storyboard.json",
            "boundary-cancellation-check.json",
            "proof-dependency-check.json",
            "prism-chain-homotopy-check.json",
            "mayer-vietoris-check.json",
            "cw-euler-check.json",
            "attachment-lab-check.json",
            "cohomology-duality-check.json",
            "final-sanity.json"
        ],
        "tables": ["triangle-boundary-matrices.csv", "cw-homology-models.csv", "attachment-lab.csv"]
    }
}
storyboard_path = save_json(storyboard, CHECKS / "visual-storyboard.json")
display_artifact(storyboard_path)


## 1. Singular Chains: Signed Boundaries

A singular `p`-simplex is a continuous map out of the standard `p`-simplex. The definition is intentionally permissive: the image can fold, collapse, or overlap itself. The algebra does not need a tidy embedded triangle; it needs a source simplex with ordered faces. A singular `p`-chain is a finite integer combination of such maps, and the boundary operator records the alternating sum of the restrictions to faces.

The first invariant scaffold is the identity `d_{p-1} d_p = 0`. Geometrically, every codimension-two face of a simplex appears twice in the boundary of the boundary, once with each sign. Algebraically, this is the cancellation that makes `im(d_{p+1})` a subgroup of `ker(d_p)`, so the quotient `H_p = ker(d_p) / im(d_{p+1})` is defined. The figure below uses one oriented triangle as a finite proxy for the singular definition. The matrix columns are oriented simplices; multiplying the edge-boundary matrix by the triangle-boundary matrix gives zero exactly.


In [ ]:
def faces(simplex: tuple[int, ...]) -> list[tuple[int, tuple[int, ...]]]:
    return [((-1) ** i, simplex[:i] + simplex[i + 1 :]) for i in range(len(simplex))]


def boundary_matrix(p_simplices: list[tuple[int, ...]], lower_simplices: list[tuple[int, ...]]) -> sp.Matrix:
    row = {simplex: i for i, simplex in enumerate(lower_simplices)}
    mat = sp.zeros(len(lower_simplices), len(p_simplices))
    for j, simplex in enumerate(p_simplices):
        for sign, face in faces(simplex):
            mat[row[face], j] += sign
    return mat


vertices = [(0,), (1,), (2,)]
edges = [(0, 1), (0, 2), (1, 2)]
triangles = [(0, 1, 2)]
d1 = boundary_matrix(edges, vertices)
d2 = boundary_matrix(triangles, edges)
boundary_product = d1 * d2

triangle_rows = []
for name, mat, row_labels, col_labels in [
    ("d1_edges_to_vertices", d1, vertices, edges),
    ("d2_triangle_to_edges", d2, edges, triangles),
    ("d1_times_d2", boundary_product, vertices, triangles),
]:
    for i in range(mat.rows):
        for j in range(mat.cols):
            triangle_rows.append({
                "matrix": name,
                "row": str(row_labels[i]),
                "column": str(col_labels[j]),
                "value": int(mat[i, j]),
            })

save_csv(triangle_rows, TABLES / "triangle-boundary-matrices.csv")
boundary_check = {
    "d1": [[int(x) for x in row] for row in d1.tolist()],
    "d2": [[int(x) for x in row] for row in d2.tolist()],
    "d1_times_d2": [[int(x) for x in row] for row in boundary_product.tolist()],
    "boundary_squared_zero": boundary_product == sp.zeros(3, 1),
    "course_helper_triangle_check": simplex_boundary_squared_zero(),
}
save_json(boundary_check, CHECKS / "boundary-cancellation-check.json")

fig, ax = plt.subplots(figsize=(7.2, 4.6))
pts = np.array([[0.0, 0.0], [1.0, 0.0], [0.36, 0.84]])
for a, b, label in [(0, 1, "[0,1]"), (1, 2, "[1,2]"), (0, 2, "[0,2]")]:
    start, end = pts[a], pts[b]
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", lw=2, color="#214E8A"))
    mid = 0.5 * (start + end)
    ax.text(mid[0], mid[1] + 0.045, label, ha="center", va="center", color="#12355B")
ax.fill(pts[:, 0], pts[:, 1], color="#9BC4E2", alpha=0.25)
for i, (x, y) in enumerate(pts):
    ax.scatter([x], [y], s=70, color="#B23A48", zorder=3)
    ax.text(x, y - 0.08, f"v{i}", ha="center", va="top")
ax.text(0.5, -0.22, r"$\partial[0,1,2]=[1,2]-[0,2]+[0,1]$", ha="center", fontsize=12)
ax.text(1.27, 0.63, r"$d_1 d_2=$" + str(boundary_product.tolist()), ha="left", va="center", fontsize=12)
ax.text(1.27, 0.48, "each vertex appears once\nwith + sign and once\nwith - sign", ha="left", va="top")
ax.set_title("Boundary cancellation for one oriented 2-simplex")
ax.set_aspect("equal")
ax.set_xlim(-0.15, 2.35)
ax.set_ylim(-0.35, 1.05)
ax.axis("off")
singular_boundary_path = save_matplotlib(fig, FIGURES / "singular-boundary-cancellation.png")
plt.close(fig)

display_artifact(singular_boundary_path, width=760)
display(Markdown(f"Boundary product `d1*d2` equals `{boundary_product.tolist()}`."))


The proof of `d*d = 0` is the first place where the singular definition looks more technical than the geometric idea. The technical face maps are there so every repeated face has a name and a sign. Once that bookkeeping is in place, the rest of homology becomes a sequence of controlled failures of exactness. If every cycle were a boundary, there would be no homology. Nonzero homology records the cycles that survive after all higher-dimensional fillings have been allowed.

Functoriality is the next scaffold. A continuous map `f: X -> Y` sends each singular simplex `sigma` in `X` to `f sigma` in `Y`. Because composing with `f` commutes with taking faces, it sends cycles to cycles and boundaries to boundaries, so it descends to homology. This is the reason homology is topological: homeomorphisms give inverse homology maps. The dependency graph below records the proof flow used repeatedly throughout the chapter.


In [ ]:
proof_edges = [
    ("ordered faces", "d*d=0"),
    ("d*d=0", "cycles/boundaries"),
    ("cycles/boundaries", "homology quotient"),
    ("continuous maps", "chain maps"),
    ("chain maps", "functoriality"),
    ("functoriality", "topological invariance"),
    ("prism chain homotopy", "homotopy invariance"),
    ("homotopy invariance", "contractible spaces"),
    ("H1 map from loops", "abelianization"),
    ("short exact chain sequence", "zigzag lemma"),
    ("zigzag lemma", "Mayer-Vietoris"),
    ("subdivision", "small-chain replacement"),
    ("small-chain replacement", "Mayer-Vietoris"),
    ("Mayer-Vietoris", "sphere homology"),
    ("Mayer-Vietoris", "CW attaching"),
    ("CW attaching", "CW homology"),
    ("CW homology", "Euler characteristic"),
    ("dual Hom(-,G)", "cohomology"),
    ("homology quotient", "cohomology"),
]
G = nx.DiGraph(proof_edges)
proof_check = {
    "node_count": G.number_of_nodes(),
    "edge_count": G.number_of_edges(),
    "is_directed_acyclic": nx.is_directed_acyclic_graph(G),
    "required_targets_reached": {target: nx.has_path(G, "ordered faces", target) or target in G for target in ["homotopy invariance", "Mayer-Vietoris", "CW homology", "Euler characteristic", "cohomology"]},
}
save_json(proof_check, CHECKS / "proof-dependency-check.json")

fig, ax = plt.subplots(figsize=(10.5, 6.8))
pos = nx.spring_layout(G, seed=13, k=0.72)
colors = []
for node in G.nodes:
    if node in {"homotopy invariance", "Mayer-Vietoris", "CW homology", "Euler characteristic", "cohomology"}:
        colors.append("#F2B138")
    elif "chain" in node or "homology" in node:
        colors.append("#9BC4E2")
    else:
        colors.append("#D9E6A8")
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=12, edge_color="#555555", width=1.2)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=1750, edgecolors="#333333", linewidths=0.8)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
ax.set_title("Proof dependencies in Chapter 13 homology")
ax.axis("off")
proof_graph_path = save_matplotlib(fig, FIGURES / "homology-proof-dependency-graph.png")
plt.close(fig)

display_artifact(proof_graph_path, width=900)


## 2. Homotopy Invariance: The Prism Correction

The homotopy invariance theorem says that homotopic maps induce the same homology map. The finite picture is the prism operator. If a simplex moves through a homotopy, the trace is a prism. The prism can be triangulated. Its boundary contains the final simplex, the initial simplex with the opposite sign, and side faces. The side faces are not an error; they are exactly the `h d` term in the chain-homotopy identity

`d h + h d = G - F`.

When the input chain is a cycle, `d` of that chain is zero, so the side correction disappears in homology. The final and initial chains differ by a boundary, and therefore represent the same homology class. This proof is stronger than a picture of a deformed loop: it works in every dimension because it is an identity of chain maps.


In [ ]:
def simplex_boundary_labels(simplex: tuple[str, ...], coeff: int = 1) -> dict[tuple[str, ...], int]:
    out: dict[tuple[str, ...], int] = {}
    for i in range(len(simplex)):
        face = simplex[:i] + simplex[i + 1 :]
        out[face] = out.get(face, 0) + coeff * ((-1) ** i)
    return {k: v for k, v in out.items() if v}


def add_chains(*chains: dict[tuple[str, ...], int]) -> dict[tuple[str, ...], int]:
    out: dict[tuple[str, ...], int] = {}
    for chain in chains:
        for simplex, coeff in chain.items():
            out[simplex] = out.get(simplex, 0) + coeff
    return {k: v for k, v in out.items() if v}


T0 = ("b0", "t0", "t1")
T1 = ("b0", "b1", "t1")
partial_h_edge = add_chains(simplex_boundary_labels(T0, 1), simplex_boundary_labels(T1, -1))
h_partial_edge = {("b1", "t1"): 1, ("b0", "t0"): -1}
chain_homotopy_sum = add_chains(partial_h_edge, h_partial_edge)
expected_top_minus_bottom = {("t0", "t1"): 1, ("b0", "b1"): -1}

prism_check = {
    "partial_h_edge": {str(k): v for k, v in partial_h_edge.items()},
    "h_partial_edge": {str(k): v for k, v in h_partial_edge.items()},
    "dh_plus_hd": {str(k): v for k, v in chain_homotopy_sum.items()},
    "expected_top_minus_bottom": {str(k): v for k, v in expected_top_minus_bottom.items()},
    "identity_holds": chain_homotopy_sum == expected_top_minus_bottom,
}
save_json(prism_check, CHECKS / "prism-chain-homotopy-check.json")

coords = {"b0": (0, 0, 0), "b1": (1, 0, 0), "t0": (0, 1, 0), "t1": (1, 1, 0)}
vertex_order = ["b0", "b1", "t0", "t1"]
xyz = np.array([coords[v] for v in vertex_order])
fig = go.Figure()
fig.add_trace(go.Mesh3d(
    x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
    i=[0, 0], j=[2, 1], k=[3, 3],
    color="#9BC4E2", opacity=0.55,
    name="h(edge): two signed triangles"
))
edge_segments = [("b0", "b1", "bottom"), ("t0", "t1", "top"), ("b0", "t0", "left side"), ("b1", "t1", "right side"), ("b0", "t1", "shared diagonal")]
for a, b, name in edge_segments:
    xa, ya, za = coords[a]
    xb, yb, zb = coords[b]
    fig.add_trace(go.Scatter3d(x=[xa, xb], y=[ya, yb], z=[za, zb], mode="lines+markers+text", text=[a, b], textposition="top center", line=dict(width=6), name=name))
fig.update_layout(
    title="Prism chain homotopy for one moving 1-simplex",
    scene=dict(xaxis_title="simplex coordinate", yaxis_title="homotopy time", zaxis_title="", aspectmode="data", camera=dict(eye=dict(x=1.4, y=1.2, z=1.1))),
    margin=dict(l=0, r=0, t=45, b=0),
    showlegend=True,
)
prism_html_path = save_plotly_html(fig, HTML / "prism-chain-homotopy.html")
display_artifact(prism_html_path, width=860, height=560)

display(Markdown(f"Exact prism identity holds: `{chain_homotopy_sum == expected_top_minus_bottom}`."))


In [ ]:
rank_rows = []
examples_for_flow = {
    "S1 cellular": {"C0": 1, "C1": 1, "C2": 0, "rank d1": 0, "rank d2": 0},
    "T2 cellular": {"C0": 1, "C1": 2, "C2": 1, "rank d1": 0, "rank d2": 0},
    "RP2 cellular": {"C0": 1, "C1": 1, "C2": 1, "rank d1": 0, "rank d2": 1},
}
for name, data in examples_for_flow.items():
    betti0 = data["C0"] - data["rank d1"]
    betti1 = data["C1"] - data["rank d1"] - data["rank d2"]
    betti2 = data["C2"] - data["rank d2"]
    for degree, dim, betti in [(0, data["C0"], betti0), (1, data["C1"], betti1), (2, data["C2"], betti2)]:
        rank_rows.append({"space": name, "degree": degree, "chain_rank": dim, "betti_rank": betti})
flow_df = pd.DataFrame(rank_rows)
fig = go.Figure()
for space, grp in flow_df.groupby("space"):
    fig.add_trace(go.Scatter(x=grp["degree"], y=grp["chain_rank"], mode="lines+markers", name=f"{space} chain ranks"))
    fig.add_trace(go.Bar(x=grp["degree"], y=grp["betti_rank"], name=f"{space} Betti ranks", opacity=0.55))
fig.update_layout(title="Cellular chain ranks versus surviving homology ranks", xaxis_title="degree p", yaxis_title="rank", barmode="group")
flow_html_path = save_plotly_html(fig, HTML / "chain-complex-rank-flow.html")
display_artifact(flow_html_path, width=860, height=520)


## 3. Mayer-Vietoris: A Global Cycle from Local Pieces

Mayer-Vietoris is the homology analogue of cutting a space into overlapping pieces and asking what the overlap remembers. The source theorem uses open sets `U` and `V`, a short exact sequence of chain complexes, a zigzag connecting homomorphism, and a subdivision theorem that lets singular simplices be made small enough to lie in members of the cover. The important computational message is exactness: a class disappears at one stage precisely when it came from the previous stage.

The circle cover is the smallest nontrivial example. Let `U` and `V` be two connected arcs whose union is the circle. Each arc is contractible, so neither has `H_1`. Their intersection has two components. The difference between those two overlap components lies in the kernel of the map into `H_0(U) + H_0(V)`, because both components land in the same component of `U` and the same component of `V`. Exactness says that this kernel is the image of the connecting map from `H_1(S^1)`. In plain language, the global loop is born from the failure of the two overlap components to be identified locally in a unique way.


In [ ]:
theta = np.linspace(0, 2 * np.pi, 500)
circle = np.column_stack([np.cos(theta), np.sin(theta)])
U_mask = np.cos(theta) > -0.78
V_mask = np.cos(theta) < 0.78
intersection_mask = U_mask & V_mask

mv_matrix = sp.Matrix([[1, 1], [-1, -1]])
mv_rank = mv_matrix.rank()
mv_kernel_dim = mv_matrix.cols - mv_rank
mv_check = {
    "map_H0_intersection_to_H0_U_plus_H0_V": [[int(x) for x in row] for row in mv_matrix.tolist()],
    "rank": int(mv_rank),
    "kernel_rank": int(mv_kernel_dim),
    "computed_H1_S1_rank": int(mv_kernel_dim),
    "expected_H1_S1_rank": 1,
    "exactness_snippet_passes": int(mv_kernel_dim) == 1,
}
save_json(mv_check, CHECKS / "mayer-vietoris-check.json")

fig, ax = plt.subplots(figsize=(7.4, 5.8))
ax.plot(circle[:, 0], circle[:, 1], color="#222222", lw=1.5, alpha=0.35, label="S1")
ax.scatter(circle[U_mask, 0], circle[U_mask, 1], s=12, color="#2E86AB", label="U arc samples")
ax.scatter(circle[V_mask, 0], circle[V_mask, 1], s=12, color="#C73E1D", label="V arc samples", alpha=0.8)
upper = intersection_mask & (circle[:, 1] >= 0)
lower = intersection_mask & (circle[:, 1] < 0)
ax.scatter(circle[upper, 0], circle[upper, 1], s=34, color="#F2B138", label="U cap V component A", zorder=4)
ax.scatter(circle[lower, 0], circle[lower, 1], s=34, color="#6A994E", label="U cap V component B", zorder=4)
ax.annotate("A - B is killed by inclusion\ninto U and into V", xy=(0, 0), xytext=(-1.45, 1.25), arrowprops=dict(arrowstyle="->"), ha="left")
ax.text(0, -1.38, "Exactness turns this overlap-difference into the generator of H1(S1).", ha="center")
ax.set_title("Mayer-Vietoris on a two-arc cover of the circle")
ax.set_aspect("equal")
ax.set_xlim(-1.6, 1.6)
ax.set_ylim(-1.55, 1.55)
ax.axis("off")
ax.legend(loc="upper right", fontsize=8)
mv_path = save_matplotlib(fig, FIGURES / "mayer-vietoris-circle-cover.png")
plt.close(fig)

display_artifact(mv_path, width=760)
display(Markdown(f"The overlap map has rank `{mv_rank}`, so its kernel has rank `{mv_kernel_dim}`."))


## 4. CW Homology and Euler Characteristic

Singular homology is defined with all singular simplices, but finite CW complexes often allow a much smaller computation. The chapter proves that attaching a cell affects homology only in the dimensions near the cell, and it summarizes the consequences for finite CW complexes. In a fuller algebraic topology course this becomes cellular homology: one free abelian generator for each cell, with boundary maps determined by attaching maps. The finite chain complex is much smaller than the singular chain complex, but it gives the same homology.

This perspective makes the Euler characteristic theorem feel inevitable. The alternating count of cells equals the alternating rank of homology groups. Boundaries can move rank from one degree to the next, but exact cancellation makes the alternating total stable. Torsion, such as the `Z/2` in `H_1(RP^2)`, has rank zero and therefore does not contribute to Euler characteristic. The table below uses standard cellular models: a circle, a sphere, a torus, a genus-two orientable surface, real projective plane, and complex projective 3-space. These are finite proxies for the chapter's CW results, not copied source exercises.


In [ ]:
def betti_from_chain_ranks(chain_ranks: dict[int, int], boundary_ranks: dict[int, int]) -> dict[int, int]:
    degrees = sorted(chain_ranks)
    out = {}
    for p in degrees:
        out[p] = chain_ranks.get(p, 0) - boundary_ranks.get(p, 0) - boundary_ranks.get(p + 1, 0)
    return out


cw_models = [
    {"space": "point", "cells": {0: 1}, "boundary_ranks": {}, "torsion": "none"},
    {"space": "S1", "cells": {0: 1, 1: 1}, "boundary_ranks": {1: 0}, "torsion": "none"},
    {"space": "S2", "cells": {0: 1, 2: 1}, "boundary_ranks": {1: 0, 2: 0}, "torsion": "none"},
    {"space": "T2", "cells": {0: 1, 1: 2, 2: 1}, "boundary_ranks": {1: 0, 2: 0}, "torsion": "none"},
    {"space": "genus_2_surface", "cells": {0: 1, 1: 4, 2: 1}, "boundary_ranks": {1: 0, 2: 0}, "torsion": "none"},
    {"space": "RP2", "cells": {0: 1, 1: 1, 2: 1}, "boundary_ranks": {1: 0, 2: 1}, "torsion": "H1 has Z/2"},
    {"space": "CP3", "cells": {0: 1, 2: 1, 4: 1, 6: 1}, "boundary_ranks": {1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0}, "torsion": "none"},
]
rows = []
for model in cw_models:
    max_degree = max(model["cells"])
    cells = {p: model["cells"].get(p, 0) for p in range(max_degree + 1)}
    ranks = {p: model["boundary_ranks"].get(p, 0) for p in range(max_degree + 2)}
    betti = betti_from_chain_ranks(cells, ranks)
    chi_cells = sum(((-1) ** p) * n for p, n in cells.items())
    chi_betti = sum(((-1) ** p) * b for p, b in betti.items())
    rows.append({
        "space": model["space"],
        "cell_counts": json.dumps(cells),
        "boundary_ranks": json.dumps({k: v for k, v in ranks.items() if v != 0}),
        "betti_ranks": json.dumps(betti),
        "torsion_note": model["torsion"],
        "chi_from_cells": int(chi_cells),
        "chi_from_betti": int(chi_betti),
        "chi_match": int(chi_cells == chi_betti),
    })
cw_df = pd.DataFrame(rows)
save_csv(cw_df.to_dict("records"), TABLES / "cw-homology-models.csv")
cw_check = {
    "models": rows,
    "all_euler_checks_match": bool(cw_df["chi_match"].all()),
    "rp2_boundary_matrix_d2": [[2]],
    "rp2_d2_rank": int(sp.Matrix([[2]]).rank()),
}
save_json(cw_check, CHECKS / "cw-euler-check.json")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), gridspec_kw={"width_ratios": [1.2, 1]})
x = np.arange(len(cw_df))
axes[0].bar(x - 0.18, cw_df["chi_from_cells"], width=0.36, label="cell alternating sum", color="#2E86AB")
axes[0].bar(x + 0.18, cw_df["chi_from_betti"], width=0.36, label="Betti alternating sum", color="#F2B138")
axes[0].axhline(0, color="#333333", lw=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(cw_df["space"], rotation=35, ha="right")
axes[0].set_ylabel("Euler characteristic")
axes[0].set_title("Euler characteristic: cells versus homology ranks")
axes[0].legend(fontsize=8)

rank_matrix = []
labels = []
max_p = 6
for row in rows:
    labels.append(row["space"])
    betti = json.loads(row["betti_ranks"])
    rank_matrix.append([betti.get(str(p), betti.get(p, 0)) for p in range(max_p + 1)])
im = axes[1].imshow(rank_matrix, cmap="YlGnBu", aspect="auto")
axes[1].set_xticks(range(max_p + 1))
axes[1].set_xlabel("degree p")
axes[1].set_yticks(range(len(labels)))
axes[1].set_yticklabels(labels)
axes[1].set_title("Betti rank heatmap")
for i in range(len(labels)):
    for j in range(max_p + 1):
        value = rank_matrix[i][j]
        if value:
            axes[1].text(j, i, str(value), ha="center", va="center", color="#111111")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
fig.tight_layout()
cw_dashboard_path = save_matplotlib(fig, FIGURES / "cw-euler-betti-dashboard.png")
plt.close(fig)

display_artifact(cw_dashboard_path, width=920)
display(cw_df)


## Applied Lab: One Attaching Map, Many First Homology Groups

A useful way to experiment with cellular homology is to start with a bouquet of `m` circles and attach one 2-cell. After abelianization, the attaching map contributes one integer row or column: how many times the boundary of the 2-cell winds around each 1-cell. If the vector is zero, no rank is killed; this is what happens to the commutator attaching map in the torus after abelianization. If the vector is primitive but nonzero, one free generator is killed. If the greatest common divisor is larger than one, torsion remains.

This lab is not the singular definition, but it is a faithful computational shadow of the chapter's CW discussion and of the theorem identifying `H_1` with the abelianization of the fundamental group. Change the attachment vectors and re-run the cell to see how free rank and torsion respond.


In [ ]:
def attachment_h1_summary(vector: tuple[int, ...]) -> dict[str, object]:
    m = len(vector)
    nonzero = [abs(v) for v in vector if v]
    g = math.gcd(*nonzero) if nonzero else 0
    killed_rank = 1 if g else 0
    free_rank = m - killed_rank
    torsion = "none" if g in (0, 1) else f"Z/{g}"
    return {"attachment_vector": vector, "gcd": g, "free_rank_H1": free_rank, "torsion_H1": torsion}


attachment_vectors = [
    (0, 0),
    (1, 0),
    (2, 0),
    (2, 3),
    (4, 6),
    (0, 0, 0, 0),
]
lab_rows = [attachment_h1_summary(v) for v in attachment_vectors]
lab_df = pd.DataFrame([{**row, "attachment_vector": str(row["attachment_vector"])} for row in lab_rows])
save_csv(lab_df.to_dict("records"), TABLES / "attachment-lab.csv")
lab_check = {
    "rows": lab_df.to_dict("records"),
    "torus_style_relation_keeps_rank_two": lab_rows[0]["free_rank_H1"] == 2,
    "rp2_style_relation_has_Z2": attachment_h1_summary((2,))["torsion_H1"] == "Z/2",
}
save_json(lab_check, CHECKS / "attachment-lab-check.json")

fig, ax = plt.subplots(figsize=(8.2, 4.5))
labels = lab_df["attachment_vector"].tolist()
ax.bar(np.arange(len(labels)), lab_df["free_rank_H1"], color="#6A994E")
for i, torsion in enumerate(lab_df["torsion_H1"]):
    ax.text(i, lab_df.loc[i, "free_rank_H1"] + 0.05, torsion, ha="center", va="bottom", fontsize=9)
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylim(0, max(lab_df["free_rank_H1"]) + 1)
ax.set_ylabel("free rank of H1")
ax.set_title("Effect of one 2-cell attaching vector on H1 of a bouquet")
fig.tight_layout()
lab_path = save_matplotlib(fig, FIGURES / "cw-attachment-lab.png")
plt.close(fig)

display_artifact(lab_path, width=820)
display(lab_df)


## 5. Cohomology Preview: Test Functions on Cycles

Cohomology begins by applying `Hom(-, G)` to the singular chain groups. A `p`-cochain is a homomorphism from `C_p(X)` to the coefficient group `G`; it assigns a number or group element to each `p`-chain. The boundary operator on chains becomes a coboundary operator on cochains by precomposition. Because precomposition reverses arrows, cohomology is contravariant: a map `f: X -> Y` gives a pullback `H^p(Y; G) -> H^p(X; G)`.

The chapter only previews the subject, but one result is already visible over fields of characteristic zero. If the homology groups are finitely generated, the dimension of `H^p(X; F)` equals the rank of `H_p(X)`. Torsion is invisible over such fields. The diagram below contrasts the chain complex direction with the cochain complex direction for the torus. The matrices are transposed in the finite model, and all boundary ranks are zero for the standard torus CW structure, so the cohomology dimensions match the Betti ranks `(1, 2, 1)`.


In [ ]:
torus_chain_dims = {0: 1, 1: 2, 2: 1}
torus_boundary_ranks = {1: 0, 2: 0}
torus_betti = betti_from_chain_ranks(torus_chain_dims, torus_boundary_ranks)
torus_cohom_dims = {p: torus_betti[p] for p in torus_betti}
cohomology_check = {
    "field": "Q (characteristic zero)",
    "torus_homology_betti_ranks": torus_betti,
    "torus_cohomology_dimensions": torus_cohom_dims,
    "dimensions_match": torus_betti == torus_cohom_dims,
    "rp2_over_Q_note": "The Z/2 torsion in H1(RP2) contributes no Q-linear cohomology dimension.",
}
save_json(cohomology_check, CHECKS / "cohomology-duality-check.json")

DG = nx.DiGraph()
DG.add_edges_from([("C2", "C1"), ("C1", "C0"), ("C^0", "C^1"), ("C^1", "C^2")])
pos = {"C2": (0, 1), "C1": (1.5, 1), "C0": (3, 1), "C^0": (0, 0), "C^1": (1.5, 0), "C^2": (3, 0)}
fig, ax = plt.subplots(figsize=(8.2, 3.7))
node_colors = ["#9BC4E2" if "^" not in n else "#F2B138" for n in DG.nodes]
nx.draw_networkx_edges(DG, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=2, edge_color="#333333")
nx.draw_networkx_nodes(DG, pos, node_size=1700, node_color=node_colors, edgecolors="#222222", ax=ax)
nx.draw_networkx_labels(DG, pos, ax=ax, font_size=12)
ax.text(1.5, 1.32, "chain boundary direction", ha="center", fontsize=11)
ax.text(1.5, -0.32, "cochain coboundary direction", ha="center", fontsize=11)
ax.text(3.55, 1.00, "rank H0 = 1", fontsize=9)
ax.text(3.55, 0.72, "rank H1 = 2", fontsize=9)
ax.text(3.55, 0.44, "rank H2 = 1", fontsize=9)
ax.set_title("Cohomology reverses arrows but preserves Betti dimensions over Q in this finite model")
ax.set_xlim(-0.5, 4.3)
ax.set_ylim(-0.6, 1.6)
ax.axis("off")
cohomology_path = save_matplotlib(fig, FIGURES / "cohomology-dual-arrows.png")
plt.close(fig)

display_artifact(cohomology_path, width=820)
display(Markdown(f"Torus Betti ranks and cohomology dimensions over `Q`: `{torus_betti}`."))


## Proof and Invariant Scaffolds

The notebook has used several small proof models. The triangle boundary matrix proves the local cancellation behind `d*d = 0`. The proof dependency graph keeps the global theorem flow honest: functoriality gives topological invariance, prism chain homotopy gives homotopy invariance, subdivision supports Mayer-Vietoris, and CW attaching results lead to Euler characteristic. The prism computation is an exact chain identity, not only a drawing. The Mayer-Vietoris circle computation isolates the kernel class in the disconnected overlap. The CW table checks that cellular data and homology ranks give the same Euler characteristic. The cohomology diagram checks the characteristic-zero rank statement in a finite model.

The final cell collects these checks in one place. It also verifies that every named artifact exists and has nonzero size. If a future edit changes a file name, breaks an identity, or produces a blank image, the notebook should fail near the end rather than silently presenting stale pedagogy.


In [ ]:
required_artifacts = [
    CHECKS / "visual-storyboard.json",
    CHECKS / "boundary-cancellation-check.json",
    CHECKS / "proof-dependency-check.json",
    CHECKS / "prism-chain-homotopy-check.json",
    CHECKS / "mayer-vietoris-check.json",
    CHECKS / "cw-euler-check.json",
    CHECKS / "attachment-lab-check.json",
    CHECKS / "cohomology-duality-check.json",
    FIGURES / "singular-boundary-cancellation.png",
    FIGURES / "homology-proof-dependency-graph.png",
    FIGURES / "mayer-vietoris-circle-cover.png",
    FIGURES / "cw-euler-betti-dashboard.png",
    FIGURES / "cw-attachment-lab.png",
    FIGURES / "cohomology-dual-arrows.png",
    HTML / "prism-chain-homotopy.html",
    HTML / "chain-complex-rank-flow.html",
    TABLES / "triangle-boundary-matrices.csv",
    TABLES / "cw-homology-models.csv",
    TABLES / "attachment-lab.csv",
]
assert_artifacts(required_artifacts, min_bytes=80)

with (CHECKS / "boundary-cancellation-check.json").open(encoding="utf-8") as handle:
    boundary_data = json.load(handle)
with (CHECKS / "mayer-vietoris-check.json").open(encoding="utf-8") as handle:
    mv_data = json.load(handle)
with (CHECKS / "cw-euler-check.json").open(encoding="utf-8") as handle:
    cw_data = json.load(handle)
with (CHECKS / "cohomology-duality-check.json").open(encoding="utf-8") as handle:
    cohom_data = json.load(handle)

assert boundary_data["boundary_squared_zero"]
assert boundary_data["course_helper_triangle_check"]
assert prism_check["identity_holds"]
assert mv_data["exactness_snippet_passes"]
assert cw_data["all_euler_checks_match"]
assert lab_check["torus_style_relation_keeps_rank_two"]
assert lab_check["rp2_style_relation_has_Z2"]
assert cohom_data["dimensions_match"]
assert euler_characteristic(1, 2, 1) == 0

png_stats = [image_stats(path) for path in required_artifacts if path.suffix.lower() == ".png"]
for stat in png_stats:
    assert stat["width"] >= 400 and stat["height"] >= 250
    assert stat["max_channel_stddev"] > 4.0

final_sanity = {
    "artifact_count": len(required_artifacts),
    "png_count": len(png_stats),
    "all_required_artifacts_nonempty": True,
    "boundary_squared_zero": boundary_data["boundary_squared_zero"],
    "prism_chain_homotopy_identity": prism_check["identity_holds"],
    "mayer_vietoris_H1_S1_rank": mv_data["computed_H1_S1_rank"],
    "cw_euler_checks_match": cw_data["all_euler_checks_match"],
    "cohomology_dimensions_match_over_Q": cohom_data["dimensions_match"],
    "image_stats": png_stats,
}
save_json(final_sanity, CHECKS / "final-sanity.json")
display_artifact(CHECKS / "final-sanity.json")
final_sanity


## Takeaways

1. Homology starts with a simple exact identity: the boundary of a boundary is zero. This turns chains into cycles, boundaries, and quotient groups.
2. Continuous maps act on singular simplices by composition, so homology is functorial and homeomorphism invariant.
3. Homotopic maps induce the same homology map because a triangulated prism gives a chain homotopy. For cycles, the difference between the two images is a boundary.
4. `H_0` counts path components, while `H_1` is the abelianized version of the fundamental group for path-connected spaces. Homology forgets noncommutative order but gains computability.
5. Mayer-Vietoris converts local homology and overlap homology into a global exact sequence. On the circle, the two-component overlap is exactly what creates the one-dimensional loop class.
6. Finite CW complexes are computable through cellular chain complexes. The same finite data explains the Euler characteristic formula as an alternating sum of Betti ranks.
7. Cohomology reverses arrows by applying `Hom(-, G)` to chains. Over characteristic-zero fields, finite-rank cohomology dimensions match homology Betti ranks, while torsion is not seen by rank.

The singular theory is broader than these finite models, but the models expose the chapter's core mechanism: topology becomes stable algebra when signed boundaries, exactness, and functorial maps are tracked with enough care.
